In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:

from functools import partial
from typing import List, Optional
import jax
import jax.numpy as jnp
import numpy as np

from flax import nnx

from probjax.nn import GaussianFourierEmbedding, Transformer
from probjax.nn.loss_fn.denoising import build_time_dependent_denoising_loss
from probjax.nn.nets.denoising_diffusion_model import EDM
from probjax.nn import MLP
from probjax.nn.utils import AffineFuse
from dmri.simulators import MultiCompartment, AllGaussianModels


In [3]:
AllGaussianModels.fraction_prior

Array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], dtype=float32)

In [4]:
from dmri.nn.tokenizer import DMRITokenizer

In [5]:
dmri_tokenizer = DMRITokenizer(AllGaussianModels, rngs=nnx.Rngs(0))

In [6]:
num_components = len(AllGaussianModels.model_types)
num_total_components = len(AllGaussianModels.model_types) + len(AllGaussianModels.noise_types)

In [7]:
models_mask = jax.random.bernoulli(jax.random.PRNGKey(0), p=0.5, shape=(100, num_total_components))

tokens_cfg = dmri_tokenizer.embed_cfgs(models_mask, jnp.ones((100, num_components)), tuple(AllGaussianModels.model_types), tuple(AllGaussianModels.noise_types), )

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


In [8]:
theta_dim = jnp.sum(AllGaussianModels.split_idx())
thetas = jax.random.normal(jax.random.PRNGKey(0), shape=(100, theta_dim))

In [9]:
tokens_params = jax.jit(dmri_tokenizer.embed_theta)(thetas, tokens_cfg)

In [10]:
tokens_params[0].shape

(19, 64)

In [11]:
dmri_tokenizer.decode_theta(tokens_params)

Array([[ 2.176,  0.433,  1.188, ..., -0.818, -0.058,  0.065],
       [-0.226, -0.311, -0.578, ...,  1.137,  0.085,  0.597],
       [-0.522, -0.075,  0.071, ...,  0.243,  0.356, -0.608],
       ...,
       [ 0.688, -2.436, -0.152, ..., -0.62 , -0.114,  0.719],
       [ 2.793,  0.339,  1.517, ..., -1.458,  0.146, -0.088],
       [ 1.442, -0.62 , -0.019, ...,  1.744, -0.185, -0.851]],      dtype=float32)

In [12]:
dmri_tokenizer.params_dims

(9, 1, 3, 3, 3, 4, 4, 4, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1)

In [83]:
from dmri.nn.dmri_reconstruction_model import DMRIInferenceModel, DMRIInferenceModelConfig

In [116]:
model = DMRIInferenceModel(DMRIInferenceModelConfig(AllGaussianModels), rngs=nnx.Rngs(0))

params = nnx.state(model, nnx.Param)

In [117]:
x = jax.random.normal(jax.random.PRNGKey(0), shape=(100,10))
bvals = jax.random.uniform(jax.random.PRNGKey(0), shape=(100,10), minval=0.1, maxval=1.0)
bvecs = jax.random.normal(jax.random.PRNGKey(0), shape=(100,10,3))
bvecs = bvecs / jnp.linalg.norm(bvecs, axis=-1, keepdims=True)

In [118]:
model(model_mask=models_mask, theta=thetas, x=x, bvals=bvals, bvecs=bvecs)

(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17)
y (100, 10, 64)
tokens_cfg (100, 19, 64)


(Array([[-1.073,  0.783,  0.551, ..., -0.505,  1.635,  0.6  ],
        [-0.812, -0.85 ,  0.136, ...,  2.068,  2.085,  2.099],
        [-0.485,  0.989,  0.418, ...,  2.306,  2.161,  2.108],
        ...,
        [-0.058,  1.146,  0.849, ...,  0.944,  2.107,  1.222],
        [-0.672, -0.847, -0.792, ...,  1.13 ,  1.373,  1.177],
        [-0.173,  1.143,  0.938, ...,  0.483,  2.71 ,  1.54 ]],      dtype=float32),
 Array([[-1.356, -2.298, -1.581, ..., -0.596,  0.025, -0.081],
        [ 0.851,  0.9  ,  1.999, ...,  0.826,  0.49 , -0.719],
        [ 1.771, -0.48 , -0.2  , ...,  0.176,  2.049,  0.729],
        ...,
        [ 0.153,  1.991,  0.747, ..., -0.452, -0.3  , -0.862],
        [-1.029, -1.778, -2.351, ..., -1.061,  0.84 ,  0.106],
        [-0.913, -1.326,  0.436, ...,  1.267, -0.708,  1.017]],      dtype=float32))

In [123]:
model.sample_mask(jax.random.PRNGKey(0), bvals[0], bvecs[0], x[0])

(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17)


Array([False,  True, False,  True, False, False, False,  True, False,
        True,  True,  True,  True,  True,  True, False,  True,  True],      dtype=bool)

In [125]:
model.sample_theta(jax.random.PRNGKey(0), bvals[0], bvecs[0], x[0], models_mask[0])

(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17)


Array([-0.12 , -2.199,  0.85 ,  3.999,  0.093,  0.804,  4.632, -0.043,
       -1.925, -1.831,  0.034, -1.81 , -0.401,  1.095, -1.376,  2.004,
        0.528,  1.581,  0.779, -0.089,  1.423,  2.6  ,  0.907, -1.833,
       -1.342, -3.251,  0.643, -0.801, -0.038,  2.823, -1.009,  0.399,
       -3.815,  0.261, -2.01 , -0.833, -0.504,  1.223, -4.455,  0.155,
       -1.906, -0.443,  0.16 ,  1.687,  2.288, -3.298,  1.044,  1.416,
        0.068,  0.353,  2.032,  0.124,  0.618,  0.022, -3.886,  2.78 ,
        0.065], dtype=float32)

In [120]:
model.loss_fn(params, jax.random.PRNGKey(0), models_mask, thetas, x, bvals, bvecs)

(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17)


Array([15.668, 15.418], dtype=float32)

In [89]:
# This is rejited
_f = jax.jit(jax.jacrev(model.loss_fn))

In [90]:

grad_per_loss = _f(params, jax.random.PRNGKey(0), models_mask, thetas, x, bvals, bvecs)

(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17)


In [91]:
grad1 = jax.tree_util.tree_map(lambda x: x[0], grad_per_loss)
grad2 = jax.tree_util.tree_map(lambda x: x[1], grad_per_loss)

In [99]:
jax.tree_util.tree_reduce(lambda x, y: x + y, jax.tree_util.tree_map(lambda x: jnp.sum(jnp.abs(x)), grad1.tokenizer))


Array(266.449, dtype=float32)

In [94]:
jax.tree_util.tree_reduce(
    lambda x, y: x + y,
    jax.tree_util.tree_map(lambda x: jnp.sum(jnp.abs(x)), grad2.tokenizer)
)

Array(497.194, dtype=float32)

In [162]:


def gradnorm_step(params,
                multitask_loss,
                alpha,
                batch,
                alpha_lr = 1e-3,
                reference_losses = 15.,
                R=1.0,
                eps=1e-8):
    """
    Perform one GradNorm-style step to:
      1. Compute per-task losses and gradients.
      2. Combine them (weighted by alpha).
      3. Update alpha to match the GradNorm objective.

    Args:
      params: model parameters (PyTree).
      alpha: jnp.array of shape (T,) = current per-task weights.
      alpha_lr: float = learning rate to update alpha.
      reference_losses: jnp.array of shape (T,) used to compute the ratio r_i = L_i / reference_losses[i].
      batch: your data batch.
      R: exponent in GradNorm (usually 1.0).
      eps: small constant for numerical stability.

    Returns:
      combined_grads: PyTree with the same shape as params, the weighted sum of per-task gradients.
      new_alpha: updated task weights (jnp.array of shape (T,)).
      new_reference_losses: possibly updated reference losses if you want to track them over time.
      (losses, gradnorm_obj): for logging/analysis
    """
    # 1) Evaluate the T losses
    losses = multitask_loss(params, *batch)  # shape (T,)

    # 2) Compute the T gradients (jacrev returns a PyTree with shape (T, ...) in each leaf)
    #    "jacrev" does one backward pass that tracks how each output index depends on params.
    grads = jax.jacrev(multitask_loss)(params, *batch)  # PyTree with shape (T, ...) in each leaf

    # 3) Weighted sum of the T grads => "combined_grads"
    combined_grads = weighted_sum_of_grads(grads, alpha)

    # 4) Compute the L2 norm of each scaled gradient: alpha_i * g_i
    #    We’ll produce an array scaled_norms of shape (T,).
    scaled_norms = per_task_scaled_norm(grads, alpha, eps=eps)

    # 5) Average (or sum) of scaled norms across tasks
    avg_scaled_norm = jnp.mean(scaled_norms)

    # 6) Form the ratio r_i = L_i / reference_losses[i].
    #    Often reference_losses are the initial losses L_i(0) or a running average.
    ratios = losses / (reference_losses + eps)  # shape (T,)

    # 7) The "target" for each task's scaled norm:
    #    target_i = avg_scaled_norm * ( (r_i)^R / mean( (r_j)^R ) )
    ratio_p = jnp.power(ratios, R)        # shape (T,)
    ratio_p_mean = jnp.mean(ratio_p)      # scalar
    target_norms = avg_scaled_norm * (ratio_p / (ratio_p_mean + eps))  # shape (T,)

    # 8) GradNorm objective = sum_i( | scaled_norms[i] - target_norms[i] | )
    #    We'll do L1 distance. (The paper does an L1 penalty.)
    def gradnorm_obj_fn(a):
        # We need scaled_norms under alpha => so recalc scaled norms with "a"
        s_norms = per_task_scaled_norm(grads, a, eps=eps)
        s_avg = jnp.mean(s_norms)

        # re-check ratio with the *current* alpha?
        # Actually, the ratio depends on the losses, not alpha, so that's constant in this step.
        # We typically keep "target_norms" as well. But let's do the official formula
        # from the paper. We'll do the simpler approach: keep target_norms fixed for the gradient.
        # If you want fully consistent, you'd re-derive at "a".
        # We'll keep the "target_norms" from above to match the standard approach.
        return jnp.sum(jnp.abs(s_norms - target_norms))

    # 9) Compute gradient wrt alpha
    d_alpha = jax.grad(gradnorm_obj_fn)(alpha)
    new_alpha = alpha - alpha_lr * d_alpha

    # Usually we project alpha onto nonnegative reals, and re-scale so sum(alpha)=T or sum=1
    new_alpha = jnp.clip(new_alpha, 0.0, jnp.inf)
    sum_alpha = jnp.sum(new_alpha)
    # e.g. scale so sum of alpha == T
    if sum_alpha > 1e-12:
        new_alpha = new_alpha * (len(alpha) / sum_alpha)

    # 10) Optionally update reference_losses if you want them to track the losses over time
    #     or keep them fixed if you're always dividing by the initial losses.
    #     For example, you might do:
    momentum = 0.0  # if you want no update, keep 0; or if you want a rolling average, do e.g. 0.1
    new_reference_losses = (1.0 - momentum) * reference_losses + momentum * losses

    # Final objective value
    # final_obj = gradnorm_obj_fn(new_alpha)

    return combined_grads, new_alpha, new_reference_losses, d_alpha

def weighted_sum_of_grads(grads, alpha):
    """
    grads is a PyTree; each leaf has shape (T, ...) = one gradient per task.
    alpha is shape (T,).
    Return a PyTree with the same structure, shape (...).
    """
    def combine_leaf(leaf):
        # leaf is shape (T, ...)
        # tensordot with alpha => sum_i [alpha[i]*leaf[i,...]]
        return jnp.tensordot(alpha, leaf, axes=(0, 0))
    return jax.tree_map(combine_leaf, grads)

def per_task_scaled_norm(grads, alpha, eps=1e-8):
    """
    Compute the L2 norm of alpha_i*g_i for each task i,
    returning shape (T,).
    grads is a PyTree with shape (T, ...) in each leaf.
    alpha is shape (T,).
    """
    # We'll do a vmap over i in [0..T-1].
    def scaled_norm_i(i):
        # For each leaf, pick leaf[i,...] and multiply by alpha[i]
        # Then sum up squares across all leaves/dimensions
        # Return sqrt(...) for L2 norm
        def leaf_contrib(leaf):
            g_i = leaf[i]  # shape (...)
            return jnp.sum((alpha[i] * g_i)**2)

        total_sq = jax.tree_util.tree_reduce(
            lambda acc, x: acc + leaf_contrib(x),
            grads, 0.0
        )
        return jnp.sqrt(total_sq + eps)

    T = alpha.shape[0]
    return jax.vmap(scaled_norm_i)(jnp.arange(T))


In [163]:
grad_params, new_alpha, new_reference_losses, d_alpha = gradnorm_step(params,  model.loss_fn, jnp.array([1.,1.]), (jax.random.key(0),models_mask, thetas, x, bvals, bvecs))

(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17)
(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17)


In [164]:
new_alpha

Array([0.965, 1.035], dtype=float32)

In [169]:
new_reference_losses

Array([15., 15.], dtype=float32)

In [161]:
d_alpha

Array([ 55.292, -13.946], dtype=float32)